# Asset Class Trend Following 策略回測報告 (2024)
本筆記本實作了基於 SMA 與 ROC 的動能策略，並加入停損機制、份額保留邏輯與螞蟻演算法 (ACO) 參數最佳化。

## 1. 資料讀取與清理
讀取 `個股1.xlsx` 並依照要求進行填補。

In [ ]:
import pandas as pd
import numpy as np

def load_and_clean_data(path):
    # The Excel has multi-index headers: Ticker, Name
    # Row 0: 股票代號, Row 1: 股票名稱
    # First column is '日期' under ('股票代號', '日期')
    df = pd.read_excel(path, header=[0, 1], index_col=0)
    
    # Clean the index (dates)
    # The index might be strings like '20190102收盤價'
    # Let's extract the date part if it's there, but wait, the first column is '日期'
    # Actually, the read_excel with index_col=0 might have taken the first column.
    # Let's check the index values.
    
    df.index = pd.to_datetime([str(i)[:8] for i in df.index], format='%Y%m%d', errors='coerce')
    df = df[df.index.notnull()]
    
    # De-duplicate Level 0 (Tickers)
    # Sometimes Excel adds .1, .2 to duplicate headers
    new_level0 = [str(col).split('.')[0] for col in df.columns.get_level_values(0)]
    new_level1 = [str(col) for col in df.columns.get_level_values(1)]
    
    # Re-create MultiIndex
    df.columns = pd.MultiIndex.from_tuples(zip(new_level0, new_level1), names=['Ticker', 'Name'])
    
    # Remove any completely empty columns or the '日期' column if it's there
    if ('股票代號', '日期') in df.columns:
        df = df.drop(columns=[('股票代號', '日期')])
        
    # De-duplicate columns by Ticker
    df = df.loc[:, ~df.columns.get_level_values(0).duplicated()]
    
    # Data Cleaning as per MD: ffill then bfill
    df = df.ffill().bfill()
    
    return df

if __name__ == "__main__":
    df = load_and_clean_data('個股1.xlsx')
    print(f"Data shape: {df.shape}")
    print(f"Date range: {df.index[0]} to {df.index[-1]}")
    print(f"Sample Tickers: {df.columns.get_level_values(0)[:5].tolist()}")
    df.to_pickle('cleaned_data.pkl')
    print("Saved to cleaned_data.pkl")


## 2. 策略引擎與最佳化邏輯
包含多標的與單標的回測引擎，以及 ACO 最佳化類別。

In [ ]:
import pandas as pd
import numpy as np

def run_backtest_per_asset(df, asset_params, stop_loss_type, rb_period=5, rb_offset=0, initial_capital=30_000_000):
    """
    asset_params: dict {ticker: (sma_len, roc_len, stop_loss_val)}
    stop_loss_type: 'peak' or 'ma'
    """
    prices = df.copy()
    tickers = prices.columns.get_level_values(0)
    names = prices.columns.get_level_values(1)
    ticker_to_name = dict(zip(tickers, names))
    prices.columns = tickers
    
    # Pre-calculate indicators for each asset
    sma_matrix = pd.DataFrame(index=prices.index, columns=tickers)
    roc_matrix = pd.DataFrame(index=prices.index, columns=tickers)
    ma_stop_matrix = pd.DataFrame(index=prices.index, columns=tickers)
    
    for t in tickers:
        s_len, r_len, sl_val = asset_params[t]
        sma_matrix[t] = prices[t].rolling(s_len).mean()
        roc_matrix[t] = prices[t].pct_change(r_len)
        if stop_loss_type == 'ma':
            ma_stop_matrix[t] = prices[t].rolling(int(sl_val)).mean()

    dates = prices.index
    n_days = len(dates)
    
    cash = initial_capital
    # holdings: {ticker: {'shares': float, 'max_price': float, 'entry_date': date, 'entry_price': float, 'slot': int}}
    holdings = {} 
    
    equity = pd.Series(index=dates, dtype=float)
    trade_log = []
    holdings_log = []
    
    pending_trades = [] # List of dicts: {'ticker', 'type', 'shares'/'amount', 'reason', 'momentum'}

    # Determine start index (max of all lookbacks)
    max_lookback = 0
    for t in tickers:
        s, r, sl = asset_params[t]
        max_lookback = max(max_lookback, s, r)
        if stop_loss_type == 'ma':
            max_lookback = max(max_lookback, int(sl))
            
    if max_lookback >= n_days:
        return pd.Series([initial_capital]*n_days, index=dates), [], []

    for i in range(max_lookback, n_days):
        curr_date = dates[i]
        curr_prices = prices.iloc[i]
        
        # 1. Execute pending trades (at T+1 close)
        if pending_trades:
            sells = [t for t in pending_trades if t['type'] == 'sell']
            buys = [t for t in pending_trades if t['type'] == 'buy']
            
            for trade in sells:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                shares = trade['shares']
                cash += shares * p
                entry_info = holdings.pop(ticker, None)
                if entry_info:
                    trade_log.append({
                        'Date': curr_date,
                        'Ticker': ticker,
                        'Name': ticker_to_name[ticker],
                        'Type': 'Sell',
                        'Price': p,
                        'Shares': shares,
                        'Reason': trade['reason'],
                        'Entry Date': entry_info['entry_date'],
                        'Entry Price': entry_info['entry_price'],
                        'Return': (p / entry_info['entry_price']) - 1 if entry_info['entry_price'] != 0 else 0
                    })
            
            for trade in buys:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                if p > 0:
                    amount = min(trade['amount'], cash)
                    if amount > 1e-6: # Avoid tiny trades
                        shares = amount / p
                        holdings[ticker] = {
                            'shares': shares,
                            'max_price': p,
                            'entry_date': curr_date,
                            'entry_price': p
                        }
                        cash -= amount
                        trade_log.append({
                            'Date': curr_date,
                            'Ticker': ticker,
                            'Name': ticker_to_name[ticker],
                            'Type': 'Buy',
                            'Price': p,
                            'Shares': shares,
                            'Reason': trade['reason'],
                            'Momentum_Value': trade.get('momentum', 0)
                        })
            pending_trades = []

        # 2. Update Equity and Max Price
        port_value = cash
        for ticker, info in holdings.items():
            val = info['shares'] * curr_prices[ticker]
            port_value += val
            holdings[ticker]['max_price'] = max(holdings[ticker]['max_price'], curr_prices[ticker])
        equity.iloc[i] = port_value
        
        # 3. Log holdings
        holdings_log.append({
            'Date': curr_date,
            'Holdings': {t: info['shares'] for t, info in holdings.items()},
            'Equity': port_value
        })

        # 4. Signal Generation (for execution at T+1)
        if i == n_days - 1: continue
        
        # Stop Loss (Daily check)
        current_held_tickers = list(holdings.keys())
        for ticker in current_held_tickers:
            info = holdings[ticker]
            s_len, r_len, sl_val = asset_params[ticker]
            stop_triggered = False
            reason = ""
            if stop_loss_type == 'peak':
                if curr_prices[ticker] < info['max_price'] * (1 - sl_val):
                    stop_triggered = True
                    reason = f"Peak-to-Trough Stop ({sl_val*100:.1f}%)"
            elif stop_loss_type == 'ma':
                ma_val = ma_stop_matrix.at[curr_date, ticker]
                if curr_prices[ticker] < ma_val:
                    stop_triggered = True
                    reason = f"MA Stop ({sl_val})"
            
            if stop_triggered:
                if not any(t['ticker'] == ticker and t['type'] == 'sell' for t in pending_trades):
                    pending_trades.append({'ticker': ticker, 'type': 'sell', 'shares': info['shares'], 'reason': reason})

        # Rebalance
        if (i - max_lookback) % rb_period == rb_offset:
            eligible = (prices.iloc[i] > sma_matrix.iloc[i]) & (roc_matrix.iloc[i] > 0)
            eligible_roc = roc_matrix.iloc[i][eligible].sort_values(ascending=False)
            top_3 = eligible_roc.head(3).index.tolist()
            
            # Sell those not in top 3
            for ticker in list(holdings.keys()):
                if ticker not in top_3:
                    if not any(t['ticker'] == ticker and t['type'] == 'sell' for t in pending_trades):
                        pending_trades.append({'ticker': ticker, 'type': 'sell', 'shares': holdings[ticker]['shares'], 'reason': 'Dropped from Top 3'})
            
            # Buy new ones
            to_buy = [t for t in top_3 if t not in holdings and not any(tr['ticker'] == t and tr['type'] == 'buy' for tr in pending_trades)]
            if to_buy:
                # Count current slots that will remain
                current_remains = [t for t in holdings if t in top_3 and not any(tr['ticker'] == t and tr['type'] == 'sell' for tr in pending_trades)]
                open_slots = 3 - len(current_remains)
                if open_slots > 0:
                    # Target value per slot is Equity / 3
                    amount_per_slot = port_value / 3
                    for ticker in to_buy[:open_slots]:
                        pending_trades.append({'ticker': ticker, 'type': 'buy', 'amount': amount_per_slot, 
                                               'reason': f'Top 3 ROC ({roc_matrix.at[curr_date, ticker]:.4f})',
                                               'momentum': roc_matrix.at[curr_date, ticker]})

    equity = equity.ffill().fillna(initial_capital)
    return equity, trade_log, holdings_log

def calculate_metrics(equity):
    if len(equity.dropna()) < 2: return {'CAGR': 0, 'MaxDD': 0, 'Calmar': 0, 'WinRate': 0}
    eq = equity.dropna()
    total_return = (eq.iloc[-1] / eq.iloc[0]) - 1
    days = (eq.index[-1] - eq.index[0]).days
    years = days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    drawdown = (eq / eq.cummax()) - 1
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0
    
    daily_returns = eq.pct_change().dropna()
    win_rate = (daily_returns > 0).mean()
    
    return {
        'CAGR': cagr,
        'MaxDD': max_dd,
        'Calmar': calmar,
        'WinRate': win_rate,
        'TotalReturn': total_return
    }


In [ ]:
import pandas as pd
import numpy as np
from strategy_logic import calculate_metrics
import random

def run_single_asset_backtest(prices, ticker, sma_len, roc_len, stop_loss_type, stop_loss_val, initial_capital=10_000_000):
    if isinstance(prices[ticker], pd.DataFrame):
        p = prices[ticker].iloc[:, 0].ffill().bfill()
    else:
        p = prices[ticker].ffill().bfill()
    sma = p.rolling(sma_len).mean()
    roc = p.pct_change(roc_len)
    
    ma_stop = None
    if stop_loss_type == 'ma':
        ma_stop = p.rolling(int(stop_loss_val)).mean()
        
    dates = p.index
    n = len(dates)
    cash = initial_capital
    shares = 0
    max_price = 0
    entry_price = 0
    
    equity = pd.Series(index=dates, dtype=float)
    
    pending_buy = False
    pending_sell = False
    
    start_idx = max(sma_len, roc_len)
    if stop_loss_type == 'ma':
        start_idx = max(start_idx, int(stop_loss_val))
        
    if start_idx >= n:
        return pd.Series([initial_capital]*n, index=dates)

    for i in range(start_idx, n):
        curr_p = p.iloc[i]
        
        if pending_buy:
            shares = cash / curr_p
            cash = 0
            max_price = curr_p
            entry_price = curr_p
            pending_buy = False
        elif pending_sell:
            cash = shares * curr_p
            shares = 0
            pending_sell = False
            
        curr_equity = cash + shares * curr_p
        equity.iloc[i] = curr_equity
        
        if shares > 0:
            max_price = max(max_price, curr_p)
            # Check Stop Loss
            stop = False
            if stop_loss_type == 'peak':
                if curr_p < max_price * (1 - stop_loss_val):
                    stop = True
            elif stop_loss_type == 'ma':
                if curr_p < ma_stop.iloc[i]:
                    stop = True
            
            # Check Exit Signal
            if stop or curr_p < sma.iloc[i] or roc.iloc[i] <= 0:
                pending_sell = True
        else:
            # Check Entry Signal
            if curr_p > sma.iloc[i] and roc.iloc[i] > 0:
                pending_buy = True
                
    return equity.ffill().fillna(initial_capital)

class ACOOptimizer:
    def __init__(self, prices, ticker, sma_range, roc_range, sl_range, stop_loss_type, iterations=10, ants=10):
        self.prices = prices
        self.ticker = ticker
        self.sma_range = sma_range
        self.roc_range = roc_range
        self.sl_range = sl_range
        self.stop_loss_type = stop_loss_type
        self.iterations = iterations
        self.ants = ants
        
        # Initialize pheromones
        self.ph_sma = {v: 1.0 for v in sma_range}
        self.ph_roc = {v: 1.0 for v in roc_range}
        self.ph_sl = {v: 1.0 for v in sl_range}
        
    def select(self, pheromones):
        vals = list(pheromones.keys())
        probs = list(pheromones.values())
        total = sum(probs)
        probs = [p/total for p in probs]
        return random.choices(vals, weights=probs)[0]

    def optimize(self):
        best_params = None
        best_calmar = -np.inf
        
        for _ in range(self.iterations):
            solutions = []
            for _ in range(self.ants):
                s = self.select(self.ph_sma)
                r = self.select(self.ph_roc)
                sl = self.select(self.ph_sl)
                
                eq = run_single_asset_backtest(self.prices, self.ticker, s, r, self.stop_loss_type, sl)
                metrics = calculate_metrics(eq)
                calmar = metrics['Calmar']
                
                solutions.append(((s, r, sl), calmar))
                
                if calmar > best_calmar:
                    best_calmar = calmar
                    best_params = (s, r, sl)
            
            # Update pheromones (evaporation + deposit)
            for d in [self.ph_sma, self.ph_roc, self.ph_sl]:
                for k in d: d[k] *= 0.7 # Evaporation
                
            for params, calmar in solutions:
                if calmar > 0:
                    s, r, sl = params
                    self.ph_sma[s] += calmar
                    self.ph_roc[r] += calmar
                    self.ph_sl[sl] += calmar
                    
        return best_params, best_calmar

if __name__ == "__main__":
    # Test on one ticker
    df = pd.read_pickle('cleaned_data.pkl')
    # Use last 180 days
    df_window = df.iloc[-180:]
    
    sma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120, 150, 200]
    roc_range = [10, 20, 30, 40, 60, 100, 125, 250]
    sl_range = [0.05, 0.1, 0.15, 0.2] # For 'peak'
    
    opt = ACOOptimizer(df_window, '1101', sma_range, roc_range, sl_range, 'peak', iterations=5, ants=5)
    best_p, best_c = opt.optimize()
    print(f"Best params for 1101: {best_p}, Calmar: {best_c}")


## 3. 執行最佳化與回測
針對 131 檔標的，利用最後 180 天資料進行最佳化，並在全週期執行回測。

In [ ]:
import pandas as pd
import numpy as np
import json
from strategy_logic import run_backtest_per_asset, calculate_metrics
import matplotlib.pyplot as plt

def main():
    df = pd.read_pickle('cleaned_data.pkl')
    with open('optimal_params.json', 'r') as f:
        best_params_json = json.load(f)
    
    asset_params = {}
    for t, p in best_params_json.items():
        asset_params[t] = (p['sma'], p['roc'], p['sl_val'])
        
    # Run final backtest on full period
    equity, trades, h_log = run_backtest_per_asset(df, asset_params, 'peak')
    metrics = calculate_metrics(equity)
    
    print("Final Backtest Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
        
    # Generate Excel
    trades_df = pd.DataFrame(trades)
    # Add parameter info to trades
    def add_params(row):
        t = row['Ticker']
        p = best_params_json[t]
        return f"SMA={p['sma']}, ROC={p['roc']}, SL={p['sl_val']}"
    
    trades_df['Optimal_Params'] = trades_df.apply(add_params, axis=1)
    
    # Add descriptions in Traditional Chinese
    def get_desc(row):
        if row['Type'] == 'Buy':
            return f"選取動能值(ROC)為 {row['Momentum_Value']:.4f} 的資產 {row['Name']} 進場 (SMA/ROC 均符合進場條件)"
        else:
            return f"資產 {row['Name']} 因 {row['Reason']} 出場"
    trades_df['說明'] = trades_df.apply(get_desc, axis=1)
    
    # Equity Curve
    equity_df = pd.DataFrame(equity, columns=['Equity'])
    equity_df['Drawdown'] = (equity_df['Equity'] / equity_df['Equity'].cummax()) - 1
    
    # Equity Hold
    h_rows = []
    for entry in h_log:
        date = entry['Date']
        h = entry['Holdings']
        for t, s in h.items():
            h_rows.append({'Date': date, 'Ticker': t, 'Shares': s})
    holdings_df = pd.DataFrame(h_rows)
    
    # Summary
    summary_data = {
        'Metric': list(metrics.keys()),
        'Value': list(metrics.values())
    }
    summary_df = pd.DataFrame(summary_data)
    
    with pd.ExcelWriter('trendstrategy_results_equity2024.xlsx', engine='xlsxwriter') as writer:
        trades_df.to_excel(writer, sheet_name='Trades', index=False)
        equity_df.to_excel(writer, sheet_name='Equity_Curve')
        holdings_df.to_excel(writer, sheet_name='Equity_Hold', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
    print("Excel report generated.")
    
    # Plot equity curve for later use
    plt.figure(figsize=(12, 6))
    plt.plot(equity, label='Strategy Equity')
    plt.fill_between(equity.index, equity, equity.cummax(), color='red', alpha=0.3)
    plt.title('Strategy Equity Curve & Drawdown')
    plt.xlabel('Date')
    plt.ylabel('Equity')
    plt.grid(True)
    plt.savefig('equity_curve.png')
    print("Equity curve plot saved.")

if __name__ == "__main__":
    main()
